In [ ]:
import pandas as pd
from datetime import datetime

from modules.data_managers.player_game_data import player_game_data_manager
from modules.data_managers.player_info_data import player_info_manager
import src.data_preprocessing.nba_data_processing as processing
from src.data_preparation import nba_data_helpers
import modules.models.player_props.data_processor as player_props

from modules.data_managers.team_game_data import team_game_data_manager

In [ ]:
pd.set_option("display.max_rows", 25)
pd.set_option("display.max_columns", 100)

# PLAYER GAME DATA: #

In [ ]:
player_game_data = player_game_data_manager.get_all_player_game_data()
player_game_data = player_game_data.dropna()
player_game_data

In [ ]:
player_game_data = processing.add_points_column(
    player_game_data,
    player_game_data_manager.FIELD_GOALS_MADE.category_name, 
    player_game_data_manager.THREE_POINTERS_MADE.category_name, 
    player_game_data_manager.FREE_THROWS_MADE.category_name
    )
player_game_data

In [ ]:
player_game_data = player_game_data.sort_values(by=player_game_data_manager.GAME_DATE.category_name)
player_game_data

In [ ]:
player_game_data =processing.add_rest_and_long_break_columns(
    player_game_data,
    player_game_data_manager.GAME_DATE.category_name
    )
player_game_data

In [ ]:
player_game_data = processing.add_total_rebounds_column(
    player_game_data,
    player_game_data_manager.OFFENSIVE_REBOUNDS.category_name,
    player_game_data_manager.DEFENSIVE_REBOUNDS.category_name    
    )
player_game_data

In [ ]:
new_dict = {}
for player_id, game_data in player_game_data.groupby(by=player_game_data_manager.PLAYER_ID.category_name):
    sorted_data = game_data.sort_values(by=player_game_data_manager.GAME_DATE.category_name)
    new_dict[player_id] = processing.add_rest_and_long_break_columns(
    sorted_data,
    player_game_data_manager.GAME_DATE.category_name
    )

new_dict

In [ ]:
player_game_data_manager.get_game_data_after_date(datetime(2025, 6, 1))

# TEAM GAME DATA: #

In [ ]:
team_game_data = team_game_data_manager.get_all_team_game_data()
team_game_data = team_game_data.dropna()
team_game_data

In [ ]:
team_game_data = team_game_data.sort_values(by=team_game_data_manager.GAME_DATE.category_name)
team_game_data

In [ ]:
team_game_data = processing.add_encoded_home_or_away_column(
    team_game_data,
    team_game_data_manager.MATCHUP.category_name
    )
team_game_data

In [ ]:
team_game_data = processing.add_possessions_column(
    team_game_data,
    team_game_data_manager.FIELD_GOALS_ATTEMPTED.category_name,
    team_game_data_manager.FREE_THROWS_ATTEMPTED.category_name,
    team_game_data_manager.OFFENSIVE_REBOUNDS.category_name,
    team_game_data_manager.TURNOVERS.category_name    
    )
team_game_data

In [ ]:
team_game_data = processing.add_encoded_season_type_columns(
    team_game_data,
    team_game_data_manager.GAME_ID.category_name
    )
team_game_data

In [ ]:
team_game_data = processing.add_opponent_stat_columns(
    team_game_data,
    team_game_data_manager.GAME_ID.category_name,
    team_game_data_manager.TEAM_ID.category_name,
    )
team_game_data

# PLAYER INFO: #

In [ ]:
player_info = player_info_manager.get_all_players_info()
player_info = player_info.dropna()
player_info

In [ ]:
player_info['NAME'] = player_info[player_info_manager.PLAYER_ID.category_name].apply(lambda x : nba_data_helpers.player_id_to_name(x))
player_info

In [ ]:
na_data = player_info[player_info[player_info_manager.HEIGHT.category_name].isna()]

In [ ]:
missing_data = player_info[
    (player_info[player_info_manager.HEIGHT.category_name].astype(str) == '') |
    (player_info[player_info_manager.WEIGHT.category_name].astype(str) == '') |
    (player_info[player_info_manager.POSITION.category_name].astype(str) == '') | 
    (player_info[player_info_manager.FROM_YEAR.category_name].astype(str) == '') | 
    (player_info[player_info_manager.TO_YEAR.category_name].astype(str) == '')
]
missing_data

In [ ]:
missing_player_ids = set(missing_data[player_info_manager.PLAYER_ID.category_name])

games_with_missing_player_info = player_game_data[
    player_game_data[player_game_data_manager.PLAYER_ID.category_name].isin(
        missing_player_ids
    )
    ]

games_with_missing_player_info

In [ ]:
invalid_weights = player_info[
    player_info[player_info_manager.WEIGHT.category_name].astype(str) == ''
]
invalid_weights

In [ ]:
player_info = processing.convert_height_to_inches(
    player_info,
    player_info_manager.HEIGHT.category_name
)
player_info

In [ ]:
positions = set(player_info[player_info_manager.POSITION.category_name])
positions

In [ ]:
player_info = processing.add_encoded_position_columns(
    player_info,
    player_info_manager.POSITION.category_name
)
player_info

In [ ]:
player_info[player_info_manager.BIRTHDATE.category_name] = pd.to_datetime(player_info[player_info_manager.BIRTHDATE.category_name])
player_info

In [ ]:
player_info[player_info_manager.FROM_YEAR.category_name] = pd.to_datetime(player_info[player_info_manager.FROM_YEAR.category_name], format='%Y')
player_info

In [ ]:
player_game_data_with_info = player_game_data.merge(
    player_info, 
    left_on=player_game_data_manager.PLAYER_ID.category_name, 
    right_on=player_info_manager.PLAYER_ID.category_name
    )
player_game_data_with_info

# NBA DATA PROCESSING #

In [ ]:
player_team_info_game_data = processing.merge_player_team_game_data(
        player_game_data_with_info,
        team_game_data,
        team_game_data_manager.GAME_ID.category_name,
        player_game_data_manager.GAME_ID.category_name,
        team_game_data_manager.TEAM_ID.category_name,
        player_game_data_manager.TEAM_ID.category_name
    )
player_team_info_game_data

In [ ]:
team_game_data = processing.add_total_player_stats_per_game_by_attribute(
        team_game_data,
        player_game_data,
        player_info,
        player_info_manager.PLAYER_ID.category_name,
        player_game_data_manager.PLAYER_ID.category_name,
        team_game_data_manager.GAME_ID.category_name,
        team_game_data_manager.TEAM_ID.category_name,
        player_props.ATTRIBUTE_COLUMNS,
        player_props.STAT_COLUMNS
    )

In [ ]:
preprocessed_data = player_props.preprocess_game_data(
    player_info,
    player_game_data,
    team_game_data,
    True
)
preprocessed_data

# PLAYER PROPS: #

In [ ]:
datasets = player_props.get_all_player_props_input_output_labeled_data(
        player_props.TargetStat.PLAYER_POINTS,
        validation_split_start_date = datetime(2024, 1, 1),
        test_split_start_date = datetime(2025, 1, 1),
        chunk_step_size = 250,
        print_debug = True
        )

In [ ]:
PLAYER_NUM = 1000

In [ ]:
datasets[0][PLAYER_NUM]['PAST_GAMES_SEQUENCE']

In [ ]:
datasets[0][PLAYER_NUM]['PAST_GAMES_AGAINST_OPPONENT_SEQUENCE']

In [ ]:
datasets[0][PLAYER_NUM]['STATIC_DATA']

In [ ]:
datasets[0][PLAYER_NUM]['PLAYER_INFO']

In [ ]:
datasets[1][PLAYER_NUM]